# Capa Gold


In [1]:
%idle_timeout 2880
%glue_version 4.0
%worker_type G.1X
%number_of_workers 5

import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
  
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.7 
Current idle_timeout is None minutes.
idle_timeout has been set to 2880 minutes.
Setting Glue version to: 4.0
Previous worker type: None
Setting new worker type to: G.1X
Previous number of workers: None
Setting new number of workers to: 5
Trying to create a Glue session for the kernel.
Session Type: glueetl
Worker Type: G.1X
Number of Workers: 5
Idle Timeout: 2880
Session ID: bcfd7c6c-b139-42e2-bc42-8c1efb0b15c4
Applying the following default arguments:
--glue_kernel_version 1.0.7
--enable-glue-datacatalog true
Waiting for session bcfd7c6c-b139-42e2-bc42-8c1efb0b15c4 to get into ready status...
Session bcfd7c6c-b139-42e2-bc42-8c1efb0b15c4 ha

#### Lectura desde Silver


In [2]:
DB_SILVER = "final_db_silver"

fact = glueContext.create_dynamic_frame.from_catalog(
    database=DB_SILVER,
    table_name="silver_fact_accidents"
).toDF()

dim_loc = glueContext.create_dynamic_frame.from_catalog(
    database=DB_SILVER,
    table_name="silver_dim_location"
).toDF()


/opt/amazon/spark/python/lib/pyspark.zip/pyspark/sql/dataframe.py:127: UserWarning: DataFrame constructor is internal. Do not directly use it.


#### KPI_1: Accidentes por Estado


In [3]:
from pyspark.sql.functions import count, avg

kpi_acc_state = (
    fact.join(dim_loc, "location_sk")
        .groupBy("state")
        .agg(
            count("*").alias("accident_cnt"),
            avg("severity").alias("avg_severity")
        )
)


#### KPI_2 Accidentes por Ciudad

In [4]:
kpi_acc_city = (
    fact.join(dim_loc, "location_sk")
        .groupBy("state","city")
        .agg(
            count("*").alias("accident_cnt"),
            avg("severity").alias("avg_severity")
        )
)


#### KPI_3 Total de Accidentes


In [5]:
kpi_acc_city = (
    fact.join(dim_loc, "location_sk")
        .groupBy("state","city")
        .agg(
            count("*").alias("accident_cnt"),
            avg("severity").alias("avg_severity")
        )
)


In [6]:
dim_time = glueContext.create_dynamic_frame.from_catalog(
    database=DB_SILVER,
    table_name="silver_dim_time"
).toDF()

dim_weather = glueContext.create_dynamic_frame.from_catalog(
    database=DB_SILVER,
    table_name="silver_dim_weather"
).toDF()


#### KPI_4 Accidentes por Hora

In [7]:
kpi_by_hour = (
    fact.join(dim_time, "time_sk")
        .groupBy("hour")
        .agg(count("*").alias("accident_cnt"))
        .orderBy("hour")
)


#### KPI_5 Accidentes por Zona Horaria

In [8]:
kpi_by_timezone = (
    fact.join(dim_loc, "location_sk")
        .groupBy("timezone")
        .agg(count("*").alias("accident_cnt"))
)


#### KPI_6 Accidentes por Clima

In [9]:
kpi_by_weather = (
    fact.join(dim_weather, "weather_sk")
        .groupBy("weather_condition")
        .agg(
            count("*").alias("accident_cnt"),
            avg("severity").alias("avg_severity")
        )
)


#### Guardamos en Gold

In [10]:
GOLD_BASE = "s3://ef-sin-bucket/gold/"

kpi_acc_state.write.mode("overwrite").parquet(GOLD_BASE + "kpi_accidents_by_state/")
kpi_acc_city.write.mode("overwrite").parquet(GOLD_BASE + "kpi_accidents_by_city/")
kpi_total.write.mode("overwrite").parquet(GOLD_BASE + "kpi_total_accidents/")
kpi_by_hour.write.mode("overwrite").parquet(GOLD_BASE + "kpi_accidents_by_hour/")
kpi_by_timezone.write.mode("overwrite").parquet(GOLD_BASE + "kpi_accidents_by_timezone/")
kpi_by_weather.write.mode("overwrite").parquet(GOLD_BASE + "kpi_accidents_by_weather/")


NameError: name 'kpi_total' is not defined
